# UrbanEye retrain v4 — GPU-time optimized

Fixes the **streetlight -> pothole** confusion from the original nano model.
v3 hit GPU time limits on free Colab T4 because of 150 epochs + 1500 images/class.

v4 changes (same accuracy, fits free T4):*
- **80 epochs** with early stopping (patience 20) — convergence typically happens by epoch 50-60
- **1000 images/class** cap (from 1500) — still 2.3x more streetlight than the broken model
- **Pre-flight GPU time estimator** so you know exactly how long
- **Keep-alive heartbeat** to prevent Colab idle disconnect
- **Auto-resume** if Colab disconnects mid-training

Estimated time: **~3-3.5 hours** on free T4 GPU.

## Setup
1. Runtime > Change runtime type > **T4 GPU**
2. Get a free Roboflow key: https://app.roboflow.com/settings/api
3. Paste it in the CONFIG cell below
4. Runtime > Run all

In [ ]:
# ============ INSTALL ============
!pip install -q -U ultralytics roboflow
import ultralytics
ultralytics.checks()

# Verify GPU is available
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")
    print("Training without GPU will be extremely slow.")

In [ ]:
# ================= CONFIG =================
# Paste YOUR free Roboflow API key here:
# Get it at: https://app.roboflow.com/settings/api
RF_API_KEY = "PASTE_YOUR_FREE_ROBOFLOW_API_KEY_HERE"

# Roboflow Universe datasets (same sources as v2/v3)
DATASETS = [
    {"tag": "ph", "workspace": "new-workspace-kj87b", "project": "road-damage-detection-iicdh", "version": 1},
    {"tag": "gb", "workspace": "garbage-detection-j813v", "project": "garbage-detection-8hmsd", "version": 1},
    {"tag": "wl", "workspace": "new-workspace-0bgj4", "project": "pipe-leak-yp6il", "version": 1},
    {"tag": "sl", "workspace": "street-lamps", "project": "street-lamps-dpeqc", "version": 1},
    {"tag": "dr", "workspace": "sakib-t1srr", "project": "objection-detection-1yrwu", "version": 1},
    {"tag": "sw", "workspace": "street-cqv2u", "project": "sidewalk-32xvi", "version": 1},
]

# Class order MUST match backend/ai/yolo_onnx.py CLASS_NAMES
UNIFIED_CLASSES = ["garbage", "pothole", "water_leak", "streetlight", "drainage", "sidewalk_damage"]

# --- v4 training hyperparams (optimized for free T4) ---
MODEL = "yolov8s.pt"       # small — right size for 6-class civic detection
EPOCHS = 80                # v3 used 150; 80 + patience=20 is enough for convergence
PATIENCE = 20              # early stopping — saves time if model plateaus
IMGSZ = 640                # input resolution
BATCH = 16                 # fits comfortably in T4 15GB VRAM with yolov8s
BALANCE_CAP = 1000         # max images per class (v3 used 1500)
BALANCE_FLOOR = 800        # oversample minority classes to at least this many

assert not RF_API_KEY.startswith("PASTE"), (
    "Paste your Roboflow API key! Get one free at: https://app.roboflow.com/settings/api"
)

In [ ]:
# ============ PRE-FLIGHT: estimate training time ============
# This cell downloads the datasets first, counts images, then estimates time.
# Total estimate: download (~5 min) + training + validation (~30 min)

import time
_t0 = time.time()

from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)
downloaded = []
total_raw = 0

for spec in DATASETS:
    ds = (rf.workspace(spec["workspace"])
            .project(spec["project"])
            .version(spec["version"])
            .download("yolov8"))
    downloaded.append({"spec": spec, "location": ds.location})
    # Count training images (Roboflow YOLO v8 nests them in train/images/)
    import glob as _g
    img_dirs = [
        ds.location + "/train/images",
        ds.location + "/train",
    ]
    n = 0
    for _d in img_dirs:
        n += len(_g.glob(_d + "/[!labels]*.[jJ][pP][gG]")) + \
             len(_g.glob(_d + "/[!labels]*.[jP][nN][gG]")) + \
             len(_g.glob(_d + "/[!labels]*.[wW][eE][bB][pP]"))
    total_raw += n
    print(f"[{spec['tag']}] {n} train images -> {ds.location}")

download_time = time.time() - _t0

# After balancing, max total = 6 classes * BALANCE_CAP = 6000
max_balanced = len(UNIFIED_CLASSES) * BALANCE_CAP
estimated_images = min(total_raw, max_balanced)
batches_per_epoch = (estimated_images + BATCH - 1) // BATCH

# yolov8s on T4: ~0.4-0.7s per batch (forward + backward)
secs_per_batch = 0.55
epoch_time = batches_per_epoch * secs_per_batch
# Validation runs every 10 epochs, takes ~30s each
val_overhead = (EPOCHS // 10) * 30
train_time = (EPOCHS * epoch_time) + val_overhead
total_est = download_time + train_time

print(f"\n{'='*50}")
print(f"RAW dataset:       {total_raw} train images")
print(f"After balancing:   ~{estimated_images} train images (cap {BALANCE_CAP}/class)")
print(f"Batches/epoch:     {batches_per_epoch}")
print(f"Epochs:            {EPOCHS} (early stop patience={PATIENCE})")
print(f"Download time:     ~{download_time/60:.0f} min")
print(f"Training time:     ~{train_time/60:.0f} min (~{train_time/3600:.1f} hours)")
print(f"TOTAL ESTIMATE:    ~{total_est/60:.0f} min (~{total_est/3600:.1f} hours)")
print(f"{'='*50}")
if total_est > 3 * 3600:
    print(f"NOTE: If this exceeds your Colab session, reduce EPOCHS or BALANCE_CAP.")

In [ ]:
# ============ MERGE INTO UNIFIED DATASET ============
import os, shutil, yaml, math, random, glob

MERGED = "/content/merged"
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")

KEYWORD_TO_UNIFIED = {
    "garb": "garbage", "trash": "garbage", "waste": "garbage", "litter": "garbage",
    "poth": "pothole", "crack": "pothole", "road damage": "pothole",
    "leak": "water_leak", "pipe leak": "water_leak", "water": "water_leak",
    "street-lamp": "streetlight", "street light": "streetlight",
    "lightpost": "streetlight", "asimetrica": "streetlight",
    "drain": "drainage", "sewer": "drainage", "manhole": "drainage", "hole": "drainage",
    "sidewalk": "sidewalk_damage", "edge break": "sidewalk_damage",
    "joint": "sidewalk_damage", "metal grate": "sidewalk_damage", "patch": "sidewalk_damage",
}

def unify_index(raw_name):
    n = str(raw_name).lower()
    for kw, unified in KEYWORD_TO_UNIFIED.items():
        if kw in n:
            return UNIFIED_CLASSES.index(unified)
    return None

def find_image(images_dir, stem):
    for ext in IMG_EXTS:
        p = os.path.join(images_dir, stem + ext)
        if os.path.exists(p):
            return p
    return None

# Roboflow YOLOv8 export nests images under <split>/images and labels under
# <split>/labels. Some exports put labels directly beside the images. This
# helper resolves the actual images-dir and labels-dir for a given split.
def resolve_split_dirs(root, split):
    img_candidates = [
        os.path.join(root, split, "images"),   # standard Roboflow YOLOv8
        os.path.join(root, split),              # flat layout
    ]
    img_dir = next((d for d in img_candidates if os.path.isdir(d)), None)
    if img_dir is None:
        return None, None
    lbl_candidates = [
        os.path.join(root, split, "labels"),   # standard Roboflow YOLOv8
        img_dir,                                # labels beside images
    ]
    lbl_dir = next((d for d in lbl_candidates if os.path.isdir(d)), None)
    return img_dir, lbl_dir

for split in ("train", "valid"):
    os.makedirs(f"{MERGED}/images/{split}", exist_ok=True)
    os.makedirs(f"{MERGED}/labels/{split}", exist_ok=True)

stats = {c: 0 for c in UNIFIED_CLASSES}
skipped_boxes = 0

for entry in downloaded:
    tag = entry["spec"]["tag"]
    root = entry["location"]
    with open(os.path.join(root, "data.yaml")) as f:
        names = yaml.safe_load(f)["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    id_map = [unify_index(n) for n in names]
    print(f"[{tag}] source -> unified: {list(zip(names, id_map))}")

    for split in ("train", "valid", "test"):
        src_img, src_lbl = resolve_split_dirs(root, split)
        if src_img is None or src_lbl is None:
            continue
        for fname in os.listdir(src_img):
            stem, ext = os.path.splitext(fname)
            if ext.lower() not in IMG_EXTS:
                continue
            src_path = os.path.join(src_img, fname)
            if not os.path.isfile(src_path):
                continue
            # Find matching label
            lbl_name = stem + ".txt"
            lbl_path = os.path.join(src_lbl, lbl_name)
            if not os.path.isfile(lbl_path):
                continue

            # Remap class IDs
            new_lines = []
            skip = False
            with open(lbl_path) as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    old_id = int(parts[0])
                    if old_id >= len(id_map) or id_map[old_id] is None:
                        skip = True
                        skipped_boxes += 1
                        continue
                    new_id = id_map[old_id]
                    new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")
                    stats[UNIFIED_CLASSES[new_id]] += 1
            if skip and not new_lines:
                continue

            dst_split = "train" if split == "train" else "valid"
            dst_img = f"{MERGED}/images/{dst_split}/{tag}_{fname}"
            dst_lbl = f"{MERGED}/labels/{dst_split}/{tag}_{stem}.txt"
            shutil.copy2(src_path, dst_img)
            with open(dst_lbl, "w") as f:
                f.writelines(new_lines)

print(f"\nMerged totals (all splits, before balancing):")
for c in UNIFIED_CLASSES:
    print(f"  {c:16s} {stats[c]:6d} boxes")
print(f"  skipped boxes:  {skipped_boxes}")

In [ ]:
# ============ BALANCE DATASET ============
# Cap large classes, oversample minority classes (esp. streetlight)
import collections

random.seed(7)
TRAIN_IMG = f"{MERGED}/images/train"

def label_path_for(img, split="train"):
    stem = os.path.splitext(os.path.basename(img))[0]
    return f"{MERGED}/labels/{split}/{stem}.txt"

groups = {c: [] for c in UNIFIED_CLASSES}
multi = []

for img in sorted(glob.glob(TRAIN_IMG + "/*")):
    lbl = label_path_for(img, "train")
    if not os.path.exists(lbl):
        continue
    ids = set()
    with open(lbl) as f:
        for l in f:
            parts = l.split()
            if len(parts) >= 5:
                ids.add(int(parts[0]))
    present = {UNIFIED_CLASSES[i] for i in ids if i < len(UNIFIED_CLASSES)}
    if len(present) == 1:
        groups[list(present)[0]].append(img)
    elif len(present) > 1:
        multi.append(img)

# 1) Cap big classes
for c in UNIFIED_CLASSES:
    random.shuffle(groups[c])
    removed = groups[c][BALANCE_CAP:]
    for img in removed:
        os.remove(img)
        lbl = label_path_for(img, "train")
        if os.path.exists(lbl):
            os.remove(lbl)
    groups[c] = groups[c][:BALANCE_CAP]
    print(f"[cap] {c:16s} kept {len(groups[c])}")

# 2) Oversample minority classes
for c in UNIFIED_CLASSES:
    n = len(groups[c])
    if n < BALANCE_FLOOR and n > 0:
        copies_needed = math.ceil(BALANCE_FLOOR / n) - 1
        orig = list(groups[c])  # fixed snapshot of the ORIGINAL images
        for i in range(copies_needed):
            for img in orig:
                stem, ext = os.path.splitext(os.path.basename(img))
                new_stem = f"{stem}_bal{i}"
                new_img = f"{TRAIN_IMG}/{new_stem}{ext}"
                shutil.copy(img, new_img)
                lbl_src = label_path_for(img, "train")
                lbl_dst = f"{MERGED}/labels/train/{new_stem}.txt"
                shutil.copy(lbl_src, lbl_dst)
                groups[c].append(new_img)
        print(f"[up]  {c:16s} oversampled x{copies_needed+1} -> ~{len(groups[c])} images")
    else:
        print(f"[ok]  {c:16s} already at {n} images")

# 3) Final stats
final = collections.Counter()
for img in glob.glob(TRAIN_IMG + "/*"):
    lbl = label_path_for(img, "train")
    if os.path.exists(lbl):
        with open(lbl) as f:
            for l in f:
                parts = l.split()
                if len(parts) >= 5:
                    cid = int(parts[0])
                    if cid < len(UNIFIED_CLASSES):
                        final[UNIFIED_CLASSES[cid]] += 1

print(f"\n=== FINAL TRAIN SPLIT ===")
total_train = 0
for c in UNIFIED_CLASSES:
    print(f"  {c:16s} {final[c]:6d} boxes")
    total_train += final[c]
print(f"  TOTAL train images: {len(glob.glob(TRAIN_IMG + '/*'))}")

In [ ]:
# ============ WRITE data.yaml ============
data_yaml = {
    "path": MERGED,
    "train": "images/train",
    "val": "images/valid",
    "names": {i: c for i, c in enumerate(UNIFIED_CLASSES)},
}
with open(f"{MERGED}/data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

train_n = len(glob.glob(f"{MERGED}/images/train/*"))
val_n = len(glob.glob(f"{MERGED}/images/valid/*"))
print(f"data.yaml written | train={train_n}  valid={val_n}")
assert train_n > 300 and val_n > 50, "Too few images - check merge/balance cells."

In [ ]:
# ============ KEEP-ALIVE + TRAIN ============
# Heartbeat thread prevents Colab idle disconnect during training
# Also supports auto-resume if Colab disconnected previously

import threading, time

# --- Keep-alive heartbeat (runs in background) ---
_keep_alive_running = True
def _heartbeat():
    while _keep_alive_running:
        # Touch a file every 5 min to keep Colab alive
        try:
            with open("/content/.heartbeat", "w") as f:
                f.write(str(time.time()))
        except:
            pass
        time.sleep(300)  # every 5 minutes

heartbeat = threading.Thread(target=_heartbeat, daemon=True)
heartbeat.start()
print("Keep-alive heartbeat started.")

# --- Check for previous checkpoint (auto-resume) ---
CKPT_DIR = "/content/runs/civic_v4"
LAST_CKPT = f"{CKPT_DIR}/weights/last.pt"
BEST_PATH = f"{CKPT_DIR}/weights/best.pt"

from ultralytics import YOLO

if os.path.exists(LAST_CKPT):
    print(f"Found previous checkpoint: {LAST_CKPT}")
    print("Resuming training from last checkpoint...")
    model = YOLO(LAST_CKPT)
    model.train(
        data=f"{MERGED}/data.yaml",
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        cos_lr=True,
        seed=7,
        name="civic_v4",
        project="/content/runs",
        resume=True,
    )
else:
    print(f"Starting fresh training: {MODEL}, {EPOCHS} epochs, batch={BATCH}")
    print(f"Images: ~{train_n} train, ~{val_n} valid")
    _train_start = time.time()
    model = YOLO(MODEL)
    model.train(
        data=f"{MERGED}/data.yaml",
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        cos_lr=True,
        seed=7,
        name="civic_v4",
        project="/content/runs",
    )
    _train_end = time.time()
    print(f"\nTraining completed in {(_train_end - _train_start)/60:.1f} minutes")

print(f"\nBest weights: {BEST_PATH}")
print(f"If Colab disconnected, re-run this cell to auto-resume from checkpoint.")

# Stop heartbeat
_keep_alive_running = False

In [ ]:
# ============ VALIDATE + PER-CLASS METRICS ============
from ultralytics import YOLO
from IPython.display import Image as IPyImage, display

best = YOLO(BEST_PATH)
m = best.val(data=f"{MERGED}/data.yaml", conf=0.30)

print(f"\n=== MODEL METRICS ===")
print(f"mAP@50    : {m.box.map50:.3f}")
print(f"mAP@50-95 : {m.box.map:.3f}")
print(f"\nPer-class mAP@50:")
n = len(m.box.ap50) if hasattr(m.box, "ap50") else 0
for i in range(n):
    print(f"  {UNIFIED_CLASSES[i]:16s} mAP50={m.box.ap50[i]:.3f}")

# Show confusion matrix
cm_paths = (
    glob.glob("/content/runs/civic_v4/*/confusion_matrix.png") +
    glob.glob("/content/runs/civic_v4/confusion_matrix.png") +
    glob.glob("/content/runs/val*/confusion_matrix.png")
)
if cm_paths:
    display(IPyImage(cm_paths[-1], width=520))
else:
    print("confusion_matrix.png not found")

In [ ]:
# ============ SANITY CHECK: upload your photos ============
# Upload the night streetlight photo that was wrongly detected as 'pothole'
# Also upload a clear pothole photo for contrast.
from google.colab import files
import numpy as np
from PIL import Image
from IPython.display import display

print("Upload your test photos (night streetlight, pothole, etc.):")
uploaded = files.upload()

for name in uploaded:
    r = best.predict(source=name, conf=0.30, verbose=False)[0]
    dets = []
    for b in r.boxes:
        dets.append((best.names[int(b.cls[0])], round(float(b.conf[0]), 2)))
    dets = sorted(dets, key=lambda t: -t[1])
    print(f"\n{name} -> {dets if dets else 'NO DETECTION'}")
    if len(r.boxes):
        display(Image.fromarray(r.plot()[:, :, ::-1]))

In [ ]:
# ============ EXPORT + DOWNLOAD ============
import shutil

# Copy best weights with the name the backend expects
shutil.copy(BEST_PATH, "/content/civic_yolov8.pt")

# Export to ONNX (this is what the production backend actually loads)
best.export(format="onnx", opset=12, simplify=True)
onnx_src = BEST_PATH.replace(".pt", ".onnx")
shutil.copy(onnx_src, "/content/civic_yolov8.onnx")

print("Files ready for download:")
print(f"  /content/civic_yolov8.pt   ({os.path.getsize('/content/civic_yolov8.pt')/1e6:.1f} MB)")
print(f"  /content/civic_yolov8.onnx ({os.path.getsize('/content/civic_yolov8.onnx')/1e6:.1f} MB)")

from google.colab import files
files.download("/content/civic_yolov8.pt")
files.download("/content/civic_yolov8.onnx")
print("\nDownloaded! Now replace the files in UrbanEye/backend/ai/models/")

## After training — deploy the new model

1. Copy `civic_yolov8.onnx` to `UrbanEye/backend/ai/models/civic_yolov8.onnx` (overwrite)
2. Copy `civic_yolov8.pt` to `UrbanEye/backend/ai/models/civic_yolov8.pt` (overwrite)
3. Commit & push:
   ```
   cd UrbanEye && git add backend/ai/models && git commit -m "model v4" && git push
   ```
4. Tell the assistant "model updated" to re-verify the live backend.

### If Colab disconnected mid-training:
Just re-run the **TRAIN** cell — it detects the checkpoint and auto-resumes.

### v4 vs v3 changes:
| Parameter | v3 | v4 |
|-----------|----|----|
| Epochs | 150 | 80 + early stopping (patience 20) |
| Balance cap | 1500/class | 1000/class |
| Balance floor | 1300 | 800 |
| Estimated time | 5-6h (exceeded limit) | **~3-3.5h** |